# CS502K Assignment 3 - Magic Square Puzzle

Arrange numbers 1-9 on a 3x3 grid so every row, column, and diagonal adds up to 15.


## Framework (copied from search4e.ipynb / AIMA Python)


In [ ]:
import math
import heapq
from collections import deque
from itertools import permutations


# The base class that all puzzles inherit from (from search4e.ipynb)
class Problem(object):
    def __init__(self, initial=None, goal=None, **kwds):
        self.__dict__.update(initial=initial, goal=goal, **kwds)

    def actions(self, state):        raise NotImplementedError
    def result(self, state, action): raise NotImplementedError
    def is_goal(self, state):        return state == self.goal
    def action_cost(self, s, a, s1): return 1
    def h(self, node):               return 0

    def __str__(self):
        return '{}({!r}, {!r})'.format(type(self).__name__, self.initial, self.goal)


# A node in the search tree (from search4e.ipynb)
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.__dict__.update(state=state, parent=parent,
                             action=action, path_cost=path_cost)
    def __repr__(self): return '<{}>'.format(self.state)
    def __len__(self): return 0 if self.parent is None else (1 + len(self.parent))
    def __lt__(self, other): return self.path_cost < other.path_cost


failure = Node('failure', path_cost=math.inf)
cutoff  = Node('cutoff',  path_cost=math.inf)


# Expand a node into its children (from search4e.ipynb)
def expand(problem, node):
    s = node.state
    for action in problem.actions(s):
        s1 = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, s1)
        yield Node(s1, node, action, cost)


# Get the list of states from start to this node (from search4e.ipynb)
def path_states(node):
    if node in (cutoff, failure, None):
        return []
    return path_states(node.parent) + [node.state]


# Queue types (from search4e.ipynb)
FIFOQueue = deque
LIFOQueue = list

class PriorityQueue:
    def __init__(self, items=(), key=lambda x: x):
        self.key = key
        self.items = []
        for item in items:
            self.add(item)
    def add(self, item):
        heapq.heappush(self.items, (self.key(item), item))
    def pop(self):
        return heapq.heappop(self.items)[1]
    def __len__(self): return len(self.items)


# BFS - explores level by level, finds shortest solution (from search4e.ipynb)
def breadth_first_search(problem):
    node = Node(problem.initial)
    if problem.is_goal(problem.initial):
        return node
    frontier = FIFOQueue([node])
    reached = {problem.initial}
    while frontier:
        node = frontier.pop()
        for child in expand(problem, node):
            s = child.state
            if problem.is_goal(s):
                return child
            if s not in reached:
                reached.add(s)
                frontier.appendleft(child)
    return failure


# Helper functions
def board9(state):
    return '{} {} {}\n{} {} {}\n{} {} {}\n'.format(*state)

def hamming_distance(A, B):
    return sum(a != b for a, b in zip(A, B))


---
## Task 1 - Solve the Magic Square Puzzle (find Figure 1)

The puzzle is like the 8-puzzle but:
- No blank tile, we use numbers 1-9
- Instead of sliding tiles, we swap any two positions
- Goal is Figure 1: (4, 9, 2, 3, 5, 7, 8, 1, 6)

We do NOT override is_goal here — the default from Problem checks
state == self.goal, which is Figure 1.


In [ ]:
class MagicSquarePuzzle(Problem):

    def __init__(self, initial, goal=(4, 9, 2, 3, 5, 7, 8, 1, 6)):
        # goal = Figure 1, used by is_goal, h1 and h2
        super().__init__(initial=initial, goal=goal)

    def actions(self, state):
        # all possible swaps between two positions (36 total)
        return [(i, j) for i in range(9) for j in range(i + 1, 9)]

    def result(self, state, action):
        # swap the numbers at position i and j
        i, j = action
        s = list(state)
        s[i], s[j] = s[j], s[i]
        return tuple(s)

    # is_goal is NOT overridden — uses default: state == self.goal (Figure 1)

    def h1(self, node):
        # misplaced tiles compared to Figure 1 (from EightPuzzle)
        return hamming_distance(node.state, self.goal)

    def h2(self, node):
        # Manhattan distance to Figure 1 (from EightPuzzle, adapted for tiles 1-9)
        X = (0, 1, 2, 0, 1, 2, 0, 1, 2)  # column of each position
        Y = (0, 0, 0, 1, 1, 1, 2, 2, 2)  # row of each position
        total = 0
        for tile in range(1, 10):
            curr = node.state.index(tile)   # where is the tile now?
            goal = self.goal.index(tile)    # where should it be?
            total += abs(X[curr]-X[goal]) + abs(Y[curr]-Y[goal])
        return total

    def h(self, node):
        return self.h2(node)


In [ ]:
# Solve from a scrambled starting state
p1 = MagicSquarePuzzle(initial=(1, 2, 3, 4, 5, 6, 7, 8, 9))

solution = breadth_first_search(p1)

print("Solution found:")
print(board9(solution.state))
print("Moves needed:", len(solution))

print("Path from start to solution:")
for i, s in enumerate(path_states(solution)):
    print(f"Step {i}:")
    print(board9(s))


---
## Task 2 - Find Additional Solutions

We modify the puzzle class by overriding is_goal to accept ANY arrangement
where rows, columns, and diagonals all sum to 15 (not just Figure 1).
This lets us find all 8 valid magic squares.


In [ ]:
# Modified class — overrides is_goal to accept any magic square
class MagicSquarePuzzleV2(MagicSquarePuzzle):

    def is_goal(self, state):
        # accept ANY arrangement where all rows, cols, diags sum to 15
        return (
            state[0]+state[1]+state[2] == 15 and
            state[3]+state[4]+state[5] == 15 and
            state[6]+state[7]+state[8] == 15 and
            state[0]+state[3]+state[6] == 15 and
            state[1]+state[4]+state[7] == 15 and
            state[2]+state[5]+state[8] == 15 and
            state[0]+state[4]+state[8] == 15 and
            state[2]+state[4]+state[6] == 15
        )


# Find ALL solutions by checking every arrangement of 1-9
checker = MagicSquarePuzzleV2(initial=(1, 2, 3, 4, 5, 6, 7, 8, 9))
all_solutions = [p for p in permutations(range(1, 10)) if checker.is_goal(p)]

print(f"Total solutions found: {len(all_solutions)}\n")
for i, sol in enumerate(all_solutions, 1):
    print(f"Solution {i}: {sol}")
    print(board9(sol))


---
## Task 3 - Compare Search Algorithms

Using the modified class from Task 2 (MagicSquarePuzzleV2), we run DFS, BFS,
and best-first (with h1 and h2) on a starting state and report summary
statistics comparing the performance of each algorithm.

h1 and h2 still use Figure 1 as their reference goal (reused from EightPuzzle),
but is_goal now accepts any magic square.


### Additional search algorithms and reporting tools (copied from search4e.ipynb)

In [ ]:
# DFS with depth limit - copied from search4e.ipynb
def is_cycle(node, k=30):
    def find_cycle(ancestor, k):
        return (ancestor is not None and k > 0 and
                (ancestor.state == node.state or find_cycle(ancestor.parent, k - 1)))
    return find_cycle(node.parent, k)

def depth_limited_search(problem, limit=50):
    frontier = LIFOQueue([Node(problem.initial)])
    result = failure
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        elif len(node) >= limit:
            result = cutoff
        elif not is_cycle(node):
            for child in expand(problem, node):
                frontier.append(child)
    return result


# Best-first search - copied from search4e.ipynb
def best_first_search(problem, f):
    node = Node(problem.initial)
    frontier = PriorityQueue([node], key=f)
    reached = {problem.initial: node}
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                frontier.add(child)
    return failure


# CountCalls and report - copied from search4e.ipynb
from collections import Counter

class CountCalls:
    """Delegate all attribute gets to the object, and count them in ._counts"""
    def __init__(self, obj):
        self._object = obj
        self._counts = Counter()

    def __getattr__(self, attr):
        self._counts[attr] += 1
        return getattr(self._object, attr)


def report(searchers, problems, verbose=True):
    for searcher in searchers:
        print(searcher.__name__ + ':')
        total_counts = Counter()
        for p in problems:
            prob = CountCalls(p)
            soln = searcher(prob)
            counts = prob._counts
            counts.update(actions=len(soln), cost=soln.path_cost)
            total_counts += counts
            if verbose: report_counts(counts, str(p)[:40])
        report_counts(total_counts, 'TOTAL\n')


def report_counts(counts, name):
    print('{:9,d} nodes |{:9,d} goal |{:5.0f} cost |{:8,d} actions | {}'.format(
          counts['result'], counts['is_goal'], counts['cost'],
          counts['actions'], name))


In [ ]:
# Wrappers so each algorithm has a clear name in the report
def dfs(problem): return depth_limited_search(problem, limit=50)
dfs.__name__ = 'depth_first_search'

def best_h1(problem): return best_first_search(problem, f=lambda n: problem.h1(n))
best_h1.__name__ = 'best_first_h1'

def best_h2(problem): return best_first_search(problem, f=lambda n: problem.h2(n))
best_h2.__name__ = 'best_first_h2'

# Run report — this prints the summary statistics table automatically
report(
    [breadth_first_search, depts, best_h1, best_h2],
    [MagicSquarePuzzleV2(initial=(1, 2, 3, 4, 5, 6, 7, 8, 9))]
)


---
## Task 4 - Top Three Learnings

### Learning 1 - BFS finds the shortest solution but explores the most nodes

BFS explores level by level so it always finds the solution with the
fewest moves (cost 6). However it expanded over 1.3 million nodes to
get there. DFS was even worse — 58 million nodes and a longer path
(cost 10). Best-first h1 found a solution in only 252 nodes (cost 7)
and h2 in 720 nodes (cost 9). So BFS guarantees the optimal path
but at the highest cost in nodes explored.

### Learning 2 - h1 and h2 from EightPuzzle still help, even when the goal is a condition

h1 and h2 measure distance to Figure 1, but is_goal accepts any of
the 8 magic squares. Even though the heuristic points to just one
solution, it still guides the search toward states where tiles are
close to a valid arrangement — and that is enough to massively reduce
the number of nodes explored (252 for h1 vs 1.3 million for BFS).

However the solution found may not be the shortest path. BFS found
cost 6, while h1 found cost 7 and h2 found cost 9. The heuristic
trades optimality for speed.

A better heuristic would measure how far each row, column, and diagonal
is from summing to 15, without assuming which specific solution we
are heading toward.

### Learning 3 - How you define actions matters

In 8-puzzle the blank slides to neighbours (2-4 moves per state).
In this puzzle any two tiles can swap (36 moves per state).
More moves means more states to explore at each level, but solutions
are also closer (fewer swaps needed). BFS works well here because the
solution is only a few swaps away, but the high branching factor
(36) explains why BFS still expanded over 1 million nodes.
